<a href="https://colab.research.google.com/github/reinhardrunkat39/lumans-bliss-qr/blob/main/klasifikasipisang.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle

In [2]:
from google.colab import files

files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"reinhardrunkat","key":"6e6d3cdc51bee8c711279d5198d406b8"}'}

In [3]:
import os

os.makedirs("/root/.kaggle", exist_ok=True)

!cp kaggle.json /root/.kaggle/

!chmod 600 /root/.kaggle/kaggle.json

In [9]:
!kaggle datasets download reinhardrunkat/datasetpisang-hsv-cnn

Dataset URL: https://www.kaggle.com/datasets/reinhardrunkat/datasetpisang-hsv-cnn
License(s): apache-2.0
100% 59.5M/59.5M [00:00<00:00, 139MB/s]



In [12]:
import zipfile

zip_path = "/content/datasetpisang-hsv-cnn.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/dataset")

In [13]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

In [24]:
import random

kelas = os.listdir("/content/dataset")

print(kelas)

['dataset_pisang']


In [25]:
IMG_SIZE = 224

In [29]:
path = "/content/dataset/dataset_pisang/Validasi/matang/musa-acuminata-banana-ad75a3ca-394a-11ec-bd23-d8c4975e38aa_jpg.rf.e5fe9ee30b9fd2bdbb692e8ed827f13c.jpg"

image = cv2.imread(path)

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

print("Gambar berhasil dimuat dan dikonversi!")

Gambar berhasil dimuat dan dikonversi!


In [30]:
lower = np.array([15,40,40])

upper = np.array([45,255,255])

mask = cv2.inRange(hsv, lower, upper)

hasil = cv2.bitwise_and(image,image,mask=mask)

In [31]:
train_datagen = ImageDataGenerator(

rescale=1./255,

rotation_range=20,

zoom_range=0.2,

horizontal_flip=True,

validation_split=0.2

)

In [37]:
model = Sequential()

model.add(
Conv2D(32,(3,3),activation='relu')
)

model.add(
MaxPooling2D((2,2))
)

model.add(
Conv2D(64,(3,3),activation='relu')
)

model.add(
MaxPooling2D((2,2))
)

model.add(
Flatten()
)

model.add(
Dense(256,activation='relu')
)

model.add(
Dropout(0.5)
)

model.add(
Dense(3,activation='softmax')
)

In [40]:
model.compile(

optimizer='adam',

loss='categorical_crossentropy',

metrics=['accuracy']

)

In [41]:
dataset_dir = "/content/dataset/dataset_pisang"

train_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=20
)

Found 562 images belonging to 3 classes.
Found 140 images belonging to 3 classes.
Epoch 1/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 88s 5s/step - accuracy: 0.6512 - loss: 4.0158 - val_accuracy: 0.7143 - val_loss: 0.5873
Epoch 2/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 71s 4s/step - accuracy: 0.7117 - loss: 0.7102 - val_accuracy: 0.7143 - val_loss: 0.5611
Epoch 3/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 70s 4s/step - accuracy: 0.7011 - loss: 0.6592 - val_accuracy: 0.7143 - val_loss: 0.5375
Epoch 4/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 84s 4s/step - accuracy: 0.6815 - loss: 0.6443 - val_accuracy: 0.7429 - val_loss: 0.5113
Epoch 5/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 71s 4s/step - accuracy: 0.7117 - loss: 0.6373 - val_accuracy: 0.7357 - val_loss: 0.6117
Epoch 6/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 71s 4s/step - accuracy: 0.7153 - loss: 0.5981 - val_accuracy: 0.7357 - val_loss: 0.5579
Epoch 7/20
18/18 ━━━━━━━━━━━━━━━━━━━━ 76s 4s/step - accuracy: 0.7206 - loss: 0.5967 - val_accuracy: 0.7286 - val_loss: 0.6205
Epoch 8/20
18/18 ━━━━━━━━━━━━━━━━━━━

In [42]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.8 MB/s eta 0:00:00


In [46]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_model(hp):
    model = Sequential()

    # Tuning jumlah filter pada lapisan Conv2D pertama
    model.add(Conv2D(
        filters=hp.Int('conv_1_filter', min_value=32, max_value=128, step=32),
        kernel_size=(3, 3),
        activation='relu'
    ))
    model.add(MaxPooling2D((2, 2)))

    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Flatten())

    # Tuning jumlah unit pada lapisan Dense
    model.add(Dense(
        units=hp.Int('dense_1_units', min_value=128, max_value=512, step=128),
        activation='relu'
    ))
    model.add(Dropout(0.5))
    model.add(Dense(3, activation='softmax')) # Sesuai dengan 3 kelas dataset Anda

    # PERBAIKAN: Masukkan hp.Choice ke dalam tf.keras.optimizers.Adam
    lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr), # <--- Sekarang sudah benar
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [47]:
import keras_tuner as kt

tuner = kt.BayesianOptimization(

build_model,

objective='val_accuracy',

max_trials=20

)

In [48]:
from sklearn.metrics import (

classification_report,

confusion_matrix

)

In [49]:
model.save(

"banana_hsv_cnn_bayes.keras"

)

In [50]:
from google.colab import files

files.download(

"banana_hsv_cnn_bayes.keras"

)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>